# Ensemble: CatBoost + LightGBM + XGBoost (with Geohash Decoding)
**Goal:** Maximum accuracy using spatial features + 3-model weighted ensemble with **5-Fold Cross-Validation**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb

## 1. Geohash Decoder (no external library needed)

In [ ]:
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
_BASE32_MAP = {c: i for i, c in enumerate(_BASE32)}

def decode_geohash(gh):
    """Decode a geohash string to (latitude, longitude)."""
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    is_lon = True
    
    for char in gh:
        val = _BASE32_MAP[char]
        for bit in [16, 8, 4, 2, 1]:
            if is_lon:
                mid = (lon_range[0] + lon_range[1]) / 2
                if val & bit:
                    lon_range[0] = mid
                else:
                    lon_range[1] = mid
            else:
                mid = (lat_range[0] + lat_range[1]) / 2
                if val & bit:
                    lat_range[0] = mid
                else:
                    lat_range[1] = mid
            is_lon = not is_lon
    
    lat = (lat_range[0] + lat_range[1]) / 2
    lon = (lon_range[0] + lon_range[1]) / 2
    return lat, lon

# Quick test
print("qp02z1 ->", decode_geohash('qp02z1'))
print("qp02zt ->", decode_geohash('qp02zt'))

## 2. Load Data

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print("Train:", train_df.shape, "| Test:", test_df.shape)

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    
    # ---- GEOHASH -> SPATIAL FEATURES ----
    coords = df['geohash'].apply(decode_geohash)
    df['latitude'] = coords.apply(lambda x: x[0])
    df['longitude'] = coords.apply(lambda x: x[1])
    
    # Prefix hierarchy
    df['gh_prefix_3'] = df['geohash'].str[:3]
    df['gh_prefix_4'] = df['geohash'].str[:4]
    df['gh_prefix_5'] = df['geohash'].str[:5]
    
    # ---- POLYNOMIAL SPATIAL INTERACTIONS ----
    df['lat_squared'] = df['latitude'] ** 2
    df['lon_squared'] = df['longitude'] ** 2
    df['lat_lon_interaction'] = df['latitude'] * df['longitude']
    
    # ---- DISTANCE FROM CENTROID ----
    lat_center = df['latitude'].mean()
    lon_center = df['longitude'].mean()
    df['dist_from_center'] = np.sqrt(
        (df['latitude'] - lat_center) ** 2 + (df['longitude'] - lon_center) ** 2
    )
    
    # ---- TIME FEATURES ----
    df['hour'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['time_slot'] = df['hour'] * 4 + df['minute'] // 15
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    df['is_peak'] = df['hour'].apply(lambda h: 1 if h in [7,8,9,17,18,19] else 0)
    
    # ---- IMPUTE MISSING ----
    df['RoadType'].fillna('Residential', inplace=True)
    df['Weather'].fillna('Sunny', inplace=True)
    df['Temperature'].fillna(df['Temperature'].median(), inplace=True)
    
    df.drop('timestamp', axis=1, inplace=True)
    return df

# Combine train + test for consistent encoding
train_df['is_train'] = 1
test_df['is_train'] = 0
test_df['demand'] = np.nan

combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
combined = engineer_features(combined)

# ---- KMEANS SPATIAL CLUSTERS ----
N_CLUSTERS = 8
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
combined['spatial_cluster'] = kmeans.fit_predict(combined[['latitude', 'longitude']])
print(f"Created {N_CLUSTERS} spatial clusters")
print("Cluster distribution:\n", combined['spatial_cluster'].value_counts().sort_index())

# Label encode ALL categorical columns (spatial_cluster included)
cat_col_names = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
                 'gh_prefix_3', 'gh_prefix_4', 'gh_prefix_5', 'spatial_cluster']
for col in cat_col_names:
    le = LabelEncoder()
    combined[col] = le.fit_transform(combined[col].astype(str))

# Split back
train_processed = combined[combined['is_train'] == 1].drop('is_train', axis=1).reset_index(drop=True)
test_processed = combined[combined['is_train'] == 0].drop(['is_train', 'demand'], axis=1).reset_index(drop=True)

print("\nTrain processed:", train_processed.shape)
print("Test processed:", test_processed.shape)
print("\nColumns:", list(train_processed.columns))

In [ ]:
drop_cols = ['demand', 'Index']
feature_cols = [c for c in train_processed.columns if c not in drop_cols]

X = train_processed[feature_cols]
y = train_processed['demand']
X_test_final = test_processed[feature_cols]

# Show the new spatial features
spatial_feats = ['latitude', 'longitude', 'lat_squared', 'lon_squared',
                 'lat_lon_interaction', 'dist_from_center', 'spatial_cluster']
print(f"Full training set: {X.shape}")
print(f"Test set: {X_test_final.shape}")
print(f"\nAll features ({len(feature_cols)}): {feature_cols}")
print(f"\nNew spatial features: {spatial_feats}")

## 4. 5-Fold Cross-Validation (CatBoost + LightGBM + XGBoost)

Each fold:
1. Trains all 3 models on the training split
2. Stores out-of-fold (OOF) predictions for proper evaluation
3. Predicts on the test set (averaged across all folds)

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# OOF prediction arrays
oof_cat = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

# Test predictions (accumulated across folds, then averaged)
test_cat = np.zeros(len(X_test_final))
test_lgb = np.zeros(len(X_test_final))
test_xgb = np.zeros(len(X_test_final))

# Per-fold metrics
fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    print(f"\n{'='*70}")
    print(f"  FOLD {fold_idx}/{N_FOLDS}  |  Train: {len(train_idx)}  |  Val: {len(val_idx)}")
    print(f"{'='*70}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # ---- CatBoost ----
    print("\n--- CatBoost ---")
    cat_model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        cat_features=cat_col_names,
        verbose=200
    )
    cat_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat[val_idx] = cat_model.predict(X_val)
    test_cat += cat_model.predict(X_test_final) / N_FOLDS
    
    # ---- LightGBM ----
    print("\n--- LightGBM ---")
    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_col_names)
    lgb_val_ds = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_col_names, reference=lgb_train)
    
    lgb_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 127,
        'min_child_samples': 20,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1
    }
    
    lgb_model = lgb.train(
        lgb_params, lgb_train,
        num_boost_round=2000,
        valid_sets=[lgb_train, lgb_val_ds],
        valid_names=['train', 'val'],
        callbacks=[lgb.log_evaluation(200), lgb.early_stopping(100)]
    )
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(X_test_final) / N_FOLDS
    
    # ---- XGBoost ----
    print("\n--- XGBoost ---")
    xgb_model = xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        tree_method='hist',
        early_stopping_rounds=100,
        eval_metric='rmse',
        verbosity=0
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)
    oof_xgb[val_idx] = xgb_model.predict(X_val)
    test_xgb += xgb_model.predict(X_test_final) / N_FOLDS
    
    # ---- Fold metrics ----
    cat_rmse_f = np.sqrt(mean_squared_error(y_val, oof_cat[val_idx]))
    lgb_rmse_f = np.sqrt(mean_squared_error(y_val, oof_lgb[val_idx]))
    xgb_rmse_f = np.sqrt(mean_squared_error(y_val, oof_xgb[val_idx]))
    cat_r2_f = r2_score(y_val, oof_cat[val_idx])
    lgb_r2_f = r2_score(y_val, oof_lgb[val_idx])
    xgb_r2_f = r2_score(y_val, oof_xgb[val_idx])
    
    fold_metrics.append({
        'fold': fold_idx,
        'cat_rmse': cat_rmse_f, 'cat_r2': cat_r2_f,
        'lgb_rmse': lgb_rmse_f, 'lgb_r2': lgb_r2_f,
        'xgb_rmse': xgb_rmse_f, 'xgb_r2': xgb_r2_f
    })
    
    print(f"\n  Fold {fold_idx} Summary:")
    print(f"    CatBoost  -> RMSE: {cat_rmse_f:.6f} | R2: {cat_r2_f:.6f}")
    print(f"    LightGBM  -> RMSE: {lgb_rmse_f:.6f} | R2: {lgb_r2_f:.6f}")
    print(f"    XGBoost   -> RMSE: {xgb_rmse_f:.6f} | R2: {xgb_r2_f:.6f}")

print(f"\n{'='*70}")
print("  5-FOLD CROSS-VALIDATION COMPLETE")
print(f"{'='*70}")

## 5. Cross-Validation Results Summary

In [ ]:
# Per-fold results table
metrics_df = pd.DataFrame(fold_metrics)
print("Per-Fold Results:")
print(metrics_df.to_string(index=False, float_format='%.6f'))

# Overall OOF metrics (computed on ALL out-of-fold predictions)
cat_oof_rmse = np.sqrt(mean_squared_error(y, oof_cat))
lgb_oof_rmse = np.sqrt(mean_squared_error(y, oof_lgb))
xgb_oof_rmse = np.sqrt(mean_squared_error(y, oof_xgb))

cat_oof_r2 = r2_score(y, oof_cat)
lgb_oof_r2 = r2_score(y, oof_lgb)
xgb_oof_r2 = r2_score(y, oof_xgb)

print("\n" + "="*60)
print("        OVERALL OOF METRICS (5-Fold CV)")
print("="*60)
print(f"  CatBoost   ->  RMSE: {cat_oof_rmse:.6f}  |  R2: {cat_oof_r2:.6f}")
print(f"  LightGBM   ->  RMSE: {lgb_oof_rmse:.6f}  |  R2: {lgb_oof_r2:.6f}")
print(f"  XGBoost    ->  RMSE: {xgb_oof_rmse:.6f}  |  R2: {xgb_oof_r2:.6f}")
print("="*60)

## 6. Ensemble - Weighted Average (using OOF RMSE)

In [ ]:
# Weights: inverse OOF RMSE (lower error -> higher weight)
w_cat = 1 / cat_oof_rmse
w_lgb = 1 / lgb_oof_rmse
w_xgb = 1 / xgb_oof_rmse
total = w_cat + w_lgb + w_xgb
w_cat, w_lgb, w_xgb = w_cat / total, w_lgb / total, w_xgb / total

print(f"Ensemble Weights -> CatBoost: {w_cat:.4f} | LightGBM: {w_lgb:.4f} | XGBoost: {w_xgb:.4f}")

# OOF ensemble
oof_ensemble = w_cat * oof_cat + w_lgb * oof_lgb + w_xgb * oof_xgb
ens_oof_rmse = np.sqrt(mean_squared_error(y, oof_ensemble))
ens_oof_r2 = r2_score(y, oof_ensemble)

print("\n" + "="*60)
print("            FINAL MODEL COMPARISON (5-Fold CV)")
print("="*60)
print(f"  CatBoost               ->  RMSE: {cat_oof_rmse:.6f}  |  R2: {cat_oof_r2:.6f}")
print(f"  LightGBM               ->  RMSE: {lgb_oof_rmse:.6f}  |  R2: {lgb_oof_r2:.6f}")
print(f"  XGBoost                ->  RMSE: {xgb_oof_rmse:.6f}  |  R2: {xgb_oof_r2:.6f}")
print("-"*60)
print(f"  ENSEMBLE (3-model, CV) ->  RMSE: {ens_oof_rmse:.6f}  |  R2: {ens_oof_r2:.6f}")
print("="*60)

## 7. Feature Importance (from last fold's LightGBM model)

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print("LightGBM Feature Importance (gain):")
print(importance.to_string(index=False))

## 8. Predict on Test and Save

In [ ]:
# Test predictions are already averaged across 5 folds
ensemble_test_pred = w_cat * test_cat + w_lgb * test_lgb + w_xgb * test_xgb

submission = pd.DataFrame({
    'Index': test_processed['Index'].astype(int),
    'demand': ensemble_test_pred
})

submission.to_csv("prediction.csv", index=False)
print(f"Saved prediction.csv - {len(submission)} rows")
submission.head()